# 5) Initial Data Preprocessing & Dataset Documentation

Initial preprocessing of the Disease and Symptom dataset.

In [ ]:
import pandas as pd
import numpy as np
import requests
import csv
from io import StringIO


## 1. Load Dataset


In [ ]:
url = "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/main/healthcare-disease-prediction/dataset/healthcare_dataset.csv"

response = requests.get(url)
response.raise_for_status()

reader = csv.reader(StringIO(response.text))
rows = list(reader)

header = rows[0]
data_rows = rows[1:]

max_fields = max(len(row) for row in data_rows)

while len(header) < max_fields:
    header.append(f"Symptom_{len(header)}")

fixed_rows = []

for row in data_rows:
    if len(row) < max_fields:
        row = row + [""] * (max_fields - len(row))
    elif len(row) > max_fields:
        row = row[:max_fields]
    fixed_rows.append(row)

df = pd.DataFrame(fixed_rows, columns=header)

print("Dataset loaded successfully!")
print("Shape:", df.shape)


In [ ]:
df.head()


## 2. Identify Columns


In [ ]:
target_column = "Disease"

symptom_columns = [
    c for c in df.columns
    if c.startswith("Symptom_")
]

print("Target column:", target_column)
print("Number of symptom columns:", len(symptom_columns))
print("Symptom columns:", symptom_columns)


## 3. Missing Value Analysis


In [ ]:
df = df.replace(r"^\s*$", np.nan, regex=True)

missing_summary = pd.DataFrame({
    "Missing Values": df.isnull().sum()
})

missing_summary["Percentage"] = (
    missing_summary["Missing Values"] / len(df) * 100
).round(2)

missing_summary


## 4. Standardize Text Values


In [ ]:
text_columns = df.select_dtypes(include="object").columns.tolist()

for column in text_columns:
    df[column] = df[column].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print("Whitespace standardized.")


In [ ]:
df[target_column] = df[target_column].apply(
    lambda x: x.title() if isinstance(x, str) else x
)

for column in symptom_columns:
    df[column] = df[column].apply(
        lambda x: x.title() if isinstance(x, str) else x
    )

print("Disease and symptom capitalization standardized.")


## 5. Duplicate Records


In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate records:", duplicate_count)

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)

print("Records before removal:", before)
print("Records after removal:", after)
print("Duplicates removed:", before - after)


## 6. Incomplete Symptom Records


In [ ]:
symptom_count = df[symptom_columns].notna().sum(axis=1)

print("Minimum symptoms per record:", symptom_count.min())
print("Maximum symptoms per record:", symptom_count.max())
print("Average symptoms per record:", round(symptom_count.mean(), 2))


In [ ]:
incomplete_records = df[symptom_count < len(symptom_columns)]
complete_records = df[symptom_count == len(symptom_columns)]

print("Complete records:", len(complete_records))
print("Incomplete records:", len(incomplete_records))
print(
    "Incomplete percentage:",
    round(len(incomplete_records) / len(df) * 100, 2),
    "%"
)


In [ ]:
incomplete_records.head(10)


## 7. Target Variable Validation


In [ ]:
print("Target variable:", target_column)
print("Number of disease classes:", df[target_column].nunique())
print("Missing target values:", df[target_column].isnull().sum())

df[target_column].value_counts().head(20)


## 8. Dataset Documentation


In [ ]:
documentation = pd.DataFrame({
    "Column": df.columns,
    "Role": [
        "Target" if c == target_column else "Feature"
        for c in df.columns
    ],
    "Data Type": [
        str(df[c].dtype)
        for c in df.columns
    ],
    "Unique Values": [
        df[c].nunique(dropna=True)
        for c in df.columns
    ],
    "Missing Values": [
        df[c].isnull().sum()
        for c in df.columns
    ]
})

documentation


## 9. Final Validation


In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Disease classes:", df[target_column].nunique())
print("Symptom columns:", len(symptom_columns))
print("Missing cells:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())


## 10. Save Cleaned Dataset


In [ ]:
output_file = "cleaned_dataset.csv"

df.to_csv(output_file, index=False)

print("Dataset saved successfully!")
print("File:", output_file)


# Conclusion

Initial data preprocessing and dataset documentation were completed. Empty values were converted to NaN, text values were standardized, duplicate records were removed, and incomplete symptom records were identified and retained.

The processed dataset is ready for further feature engineering and machine learning preprocessing.